<a href="https://colab.research.google.com/github/yoshiagent/automaticControl-colab/blob/main/notebooks/RootLocus_PID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 根軌跡與 P／I／D 控制器：主極點、P、I、PD 控制

對照網頁：[自動控制｜Root Locus Design](https://yoshiagent.github.io/automaticControl/RL-RootLocusDesign.html)

原課程用 6 個 Simulink 模型（`.mdl`）搭配 `rlocus` 說明主極點與 P／I／D 控制器對根軌跡的影響。
這本筆記本把每個模型的方塊與參數（直接從 `.mdl` 檔讀出，沒寫的參數用 Simulink 預設值：Gain＝1、Step 在 $t=1$ 由 0 跳到 1）
改寫成 Python，並畫出對應的根軌跡（MATLAB `rlocus` 的功能：$1+KG(s)=0$ 的根隨 $K$ 變化）。

| 節 | 原檔 | 內容 |
|---|---|---|
| 1 | `DominantPole.mdl` | 主極點：遠離虛軸的極點影響很小 |
| 2 | `Pcontrol.mdl` | P 控制 $\dfrac{K}{s^2+s+2}$ 與根軌跡 |
| 3 | `Icontrol.mdl` | I 控制：$K=2$ 正好在虛軸上 |
| 4 | `PDEffect.mdl` | P 與 D 動作對同一個誤差訊號的反應 |
| 5 | `Dcontrol1.mdl` | 純 D 控制 |
| 6 | `PDcontrol1.mdl` | PD 控制：加零點把根軌跡往左拉 |

只用 Colab 內建的 numpy、scipy、plotly。圖上的文字用英文。

In [1]:
import numpy as np
from scipy import signal
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def step(t, t0=1.0, after=1.0, before=0.0):
    """Simulink 的 Step 方塊（預設 Step time = 1、Initial value = 0、Final value = 1）"""
    return after if t >= t0 else before


class TF:
    """Simulink 的 Transfer Fcn 方塊：num、den 是 s 多項式係數（高次在前，同 MATLAB）。
    內部轉成狀態空間 x' = Ax + Bu、y = Cx + Du；n 是狀態數（＝分母階數）。"""
    def __init__(self, num, den):
        A, B, C, D = signal.tf2ss(num, den)
        self.A, self.B, self.C, self.D = A, B[:, 0], C[0], D[0, 0]
        self.n = len(A)

    def dx(self, x, u):
        return self.A @ x + self.B * u

    def y(self, x, u=0.0):
        return self.C @ x + self.D * u

    def dy(self, x, u=0.0):
        """輸出的微分 y' = C x'（D = 0 時成立；x 可以是整段時間的狀態陣列）"""
        Bu = self.B[:, None] * u if np.ndim(x) == 2 else self.B * u
        return self.C @ (self.A @ x + Bu)


def simulate(rhs, x0, T, breaks=(), dt=0.01, kicks=None, max_step=0.05):
    """仿 Simulink 從 t = 0 跑到 T（ode45 對應 RK45）。
    breaks：輸入跳變的時刻，在這些點分段積分，跳變前那一段一律用跳變前的輸入值；
    kicks：{時刻: 狀態增量}，表示 Derivative 方塊遇到跳變輸入時產生的脈衝（見第 5 節）。
    回傳等間隔 dt 的 t 與狀態陣列 X（X[i] 是第 i 個狀態的時間序列）。"""
    n = round(T / dt)
    t = np.linspace(0, T, n + 1)
    cuts = [0.0] + sorted(float(b) for b in breaks if 0 < b < T) + [float(T)]
    x = np.array(x0, dtype=float)
    X = np.empty((len(x), n + 1))
    for a, b in zip(cuts[:-1], cuts[1:]):
        if kicks and a in kicks:
            x = x + np.asarray(kicks[a], dtype=float)
        tb = np.nextafter(b, a)
        sol = solve_ivp(lambda s, z: rhs(min(s, tb), z), (a, b), x, method="RK45",
                        rtol=1e-9, atol=1e-12, max_step=max_step, dense_output=True)
        i0, i1 = round(a / dt), round(b / dt)
        idx = slice(i0, i1 + 1 if b == T else i1)
        X[:, idx] = sol.sol(t[idx])
        x = sol.y[:, -1]
    return t, X


def scope(t, panels, title, h=None, npts=600):
    """像 Simulink 的 Scope：panels = {子圖標題: [(曲線名稱, 數列), …]}。點數太多時等距抽樣到約 npts 點（只影響畫圖）。"""
    k = max(1, int(np.ceil(len(t) / npts)))
    fig = make_subplots(rows=len(panels), cols=1, shared_xaxes=True, subplot_titles=list(panels), vertical_spacing=0.1)
    for i, traces in enumerate(panels.values(), 1):
        for name, y in traces:
            fig.add_scatter(x=np.round(t[::k], 3).tolist(), y=np.round(np.asarray(y)[::k], 4).tolist(),
                            name=name, mode="lines", row=i, col=1)
    fig.update_xaxes(title_text="t (s)", row=len(panels), col=1)
    fig.update_layout(title=title, height=h or 260 + 220 * len(panels), template="plotly_white",
                      margin=dict(l=60, r=20, t=70, b=50))
    fig.show()


def rlocus(num, den, Ks, marks=(), title=""):
    """仿 MATLAB rlocus：畫 den(s) + K·num(s) = 0 的根（K 取 Ks 各值），marks = [(K, 顏色), …] 另外標出閉迴路極點"""
    num, den = np.asarray(num, float), np.asarray(den, float)
    R = np.array([np.roots(np.polyadd(den, k * num)) for k in Ks])
    fig = go.Figure()
    fig.add_scatter(x=np.round(R.real.ravel(), 4).tolist(), y=np.round(R.imag.ravel(), 4).tolist(), mode="markers",
                    marker=dict(size=3, color="#1f77b4"), name=f"K = {Ks[0]:g} ~ {Ks[-1]:g}")
    fig.add_scatter(x=np.roots(den).real, y=np.roots(den).imag, mode="markers", name="open-loop poles",
                    marker=dict(size=10, symbol="x-thin", line=dict(width=2, color="black")))
    if len(np.trim_zeros(num, "f")) > 1:
        z = np.roots(np.trim_zeros(num, "f"))
        fig.add_scatter(x=z.real, y=z.imag, mode="markers", name="open-loop zeros",
                        marker=dict(size=10, symbol="circle-open", line=dict(width=2, color="black")))
    for k, c in marks:
        p = np.roots(np.polyadd(den, k * num))
        fig.add_scatter(x=p.real, y=p.imag, mode="markers", marker=dict(size=11, color=c, symbol="diamond"), name=f"K = {k:g}")
    fig.add_vline(x=0, line=dict(color="gray", dash="dot"))
    fig.update_layout(title=title, xaxis_title="Re", yaxis_title="Im", height=460, template="plotly_white",
                      yaxis=dict(scaleanchor="x"))
    fig.show()


def overshoot(y, final):
    return 100 * (np.max(y) - final) / final


Gp = TF([1], [1, 1, 2])                                               # 第 2～6 節的受控體 1/(s^2+s+2)

## 1. 主極點 `DominantPole.mdl`

同一個 Step（$t=1$）送進三個系統，模擬 10 秒：

* $\dfrac{10}{s+10}\cdot\dfrac{1}{s^2+0.2s+1}$：多一個遠離虛軸的極點 $-10$（DC 增益仍為 1）；
* $\dfrac{1}{s^2+0.2s+1}$：只有一對複數極點 $-0.1\pm j0.995$；
* $\dfrac{1}{s+1}\cdot\dfrac{1}{s^2+0.2s+1}$：多一個較靠近虛軸的極點 $-1$。

網頁的說明：高階系統中最靠近虛軸的一對共軛極點稱為主極點；其他極點離得夠遠時，系統響應幾乎由主極點決定。

In [2]:
G2 = TF([1], [1, 0.2, 1])
Gfar = TF([10], np.polymul([1, 10], [1, 0.2, 1]))
Gnear = TF([1], np.polymul([1, 1], [1, 0.2, 1]))
ys = {}
for name, G in [("10/(s+10) * 1/(s^2+0.2s+1)", Gfar), ("1/(s^2+0.2s+1)", G2), ("1/(s+1) * 1/(s^2+0.2s+1)", Gnear)]:
    t, X = simulate(lambda t, x: G.dx(x, step(t)), np.zeros(G.n), 10, breaks=(1,))
    ys[name] = G.C @ X
y_far, y_2, y_near = ys.values()
print(f"DominantPole: max |far-pole system - 2nd order| = {np.abs(y_far - y_2).max():.4f}")
print(f"              max |near-pole system - 2nd order| = {np.abs(y_near - y_2).max():.4f}")
scope(t, {"Scope (Mux of the three outputs)": list(ys.items())}, "DominantPole.mdl", h=440)

DominantPole: max |far-pole system - 2nd order| = 0.0858
              max |near-pole system - 2nd order| = 0.6487


加入極點 $-10$（離虛軸是主極點的 100 倍）的響應和二階系統幾乎重合；加入極點 $-1$ 則明顯變慢、過衝變小，
這時二階近似就不準了。

## 2. P 控制 `Pcontrol.mdl`

Step（$t=0$）→ Sum（$e=r-y$）→ Gain $K=10$ → $\dfrac{1}{s^2+s+2}$，模擬 10 秒。
閉迴路 $\dfrac{K}{s^2+s+2+K}$：穩態值 $\dfrac{10}{12}=0.8333$；網頁用 `rlocus(sys1)` 看 $K$ 增加時極點的位置：
實部固定在 $-0.5$，$K$ 越大虛部越大（振盪頻率越高、阻尼比越小），但永遠不會不穩定。

In [3]:
K = 10
t, X = simulate(lambda t, x: Gp.dx(x, K * (step(t, 0) - Gp.y(x))), np.zeros(2), 10)
y = Gp.C @ X
print(f"Pcontrol K = 10: final value 10/12 = {10 / 12:.4f}; y(10) = {y[-1]:.4f} (still settling, envelope e^(-0.5t)); overshoot = {overshoot(y, 10 / 12):.1f} %")
print("closed-loop poles:", 0 + np.round(np.roots([1, 1, 2 + K]), 4))
scope(t, {"Scope": [("y", y)]}, "Pcontrol.mdl: K = 10, plant 1/(s^2+s+2)", h=380)
rlocus([1], [1, 1, 2], np.linspace(0, 20, 401), marks=[(10, "#d62728")], title="rlocus(tf(1, [1 1 2]))")

Pcontrol K = 10: final value 10/12 = 0.8333; y(10) = 0.8385 (still settling, envelope e^(-0.5t)); overshoot = 63.2 %
closed-loop poles: [-0.5+3.4278j -0.5-3.4278j]


## 3. I 控制 `Icontrol.mdl`

Step（$t=0$）→ Sum → Gain $K=2$ → Integrator → $\dfrac{1}{s^2+s+2}$，模擬 30 秒。開迴路 $\dfrac{K}{s(s^2+s+2)}$
（網頁的 `sys2=tf([1],[1 1 2 0])`），特性方程式 $s^3+s^2+2s+K=0$。Routh：$1\times2>K$ 才穩定，
所以 $K=2$ 正好是根軌跡穿過虛軸的增益：$s^3+s^2+2s+2=(s+1)(s^2+2)$，極點 $-1$、$\pm j\sqrt2$，響應持續振盪。
解析解：$y=1-\tfrac23e^{-t}-\tfrac13\cos\sqrt2t-\tfrac{\sqrt2}{3}\sin\sqrt2t$。

In [4]:
K = 2


def rhs(t, s):
    x, z = s[:2], s[2]
    e = step(t, 0) - Gp.y(x)
    return np.r_[Gp.dx(x, z), K * e]


t, X = simulate(rhs, np.zeros(3), 30)
y = Gp.C @ X[:2]
w = np.sqrt(2)
y_exact = 1 - 2 / 3 * np.exp(-t) - np.cos(w * t) / 3 - np.sqrt(2) / 3 * np.sin(w * t)
print("Icontrol K = 2 poles:", 0 + np.round(np.roots([1, 1, 2, K]), 4))
print(f"Icontrol: max |y - analytic| = {np.abs(y - y_exact).max():.1e}; oscillation amplitude = {np.sqrt(1 / 9 + 2 / 9):.4f}, period = {2 * np.pi / w:.3f} s")
scope(t, {"Scope": [("y", y)]}, "Icontrol.mdl: K = 2 puts the poles on the imaginary axis", h=380)
rlocus([1], [1, 1, 2, 0], np.linspace(0, 20, 801), marks=[(2, "#d62728")], title="rlocus(tf(1, [1 1 2 0]), 0:0.1:20)")

Icontrol K = 2 poles: [ 0.+1.4142j  0.-1.4142j -1.+0.j    ]
Icontrol: max |y - analytic| = 7.1e-09; oscillation amplitude = 0.5774, period = 4.443 s


積分器在原點加了一個極點，把根軌跡往右推，增益稍大（$K>2$）就不穩定。

## 4. P 與 D 動作 `PDEffect.mdl`

沒有受控體，只看控制器對誤差的反應：設定值 1（Step，$t=0$），量測值 $1+\sin t$（Step1 加上 Sine Wave），
誤差 $e=1-(1+\sin t)=-\sin t$。Gain（P，$K_p=1$）輸出 $e$，Derivative（D）輸出 $\dot e=-\cos t$，兩者相加為 PD。
Scope 上半畫 P、PD、D，下半畫設定值與量測值。

In [5]:
t = np.linspace(0, 10, 1001)
e = -np.sin(t)
P_act, D_act = 1 * e, -np.cos(t)
print(f"PDEffect: PD = -sqrt(2) sin(t + 45 deg), max |PD| = {np.abs(P_act + D_act).max():.4f}")
scope(t, {"Mux: P, PD, D": [("P = e", P_act), ("PD = e + de/dt", P_act + D_act), ("D = de/dt", D_act)],
          "Mux1: set point and measurement": [("set point", np.ones_like(t)), ("measurement 1 + sin t", 1 + np.sin(t))]},
      "PDEffect.mdl: error e = -sin t")

PDEffect: PD = -sqrt(2) sin(t + 45 deg), max |PD| = 1.4142


D 動作比 P 動作超前 90°：誤差還沒達到峰值，D 就已經開始反應，等於「預測」誤差的走向，所以能提供阻尼。
（Simulink 的 Derivative 方塊在 $t=0$ 的輸出固定為 0，上面畫的是解析的 $\dot e$。）

## 5. 純 D 控制 `Dcontrol1.mdl`

Step（$t=1$）→ Sum → Gain（1）→ Derivative → $\dfrac{1}{s^2+s+2}$，模擬 10 秒（MaxStep 0.001）。
閉迴路 $\dfrac{Y}{R}=\dfrac{s}{s^2+2s+2}$（網頁 `sys1=tf([1 0],[1 1 2])` 的根軌跡）。

**Derivative 方塊與步階**：設定值在 $t=1$ 跳變，$\dot e$ 在那一瞬間是脈衝 $\delta(t-1)$（Simulink 以極短時間的大數值近似），
受控體因此被「踢」一下。Python 這邊用 `kicks`：$t=1$ 時讓受控體狀態瞬間增加 $B\cdot1$；其餘時間 D 項為 $-\dot y$。
結果就是 $\dfrac{1}{s^2+2s+2}$ 的脈衝響應：$y=e^{-(t-1)}\sin(t-1)$，最後回到 0 —— 純 D 控制無法讓輸出停在設定值。

In [6]:
def rhs(t, x):
    return Gp.dx(x, -Gp.dy(x))                                        # t ≠ 1 時 de/dt = −y'


t, X = simulate(rhs, np.zeros(2), 10, breaks=(1,), kicks={1.0: Gp.B * 1.0}, dt=0.001)
y = Gp.C @ X
tau = np.clip(t - 1, 0, None)
y_exact = np.where(t >= 1, np.exp(-tau) * np.sin(tau), 0)
print(f"Dcontrol1: max |y - analytic| = {np.abs(y - y_exact).max():.1e}, y(10) = {y[-1]:.4f}")
scope(t, {"Scope": [("y", y)]}, "Dcontrol1.mdl: pure derivative control", h=380)
rlocus([1, 0], [1, 1, 2], np.linspace(0, 20, 401), marks=[(1, "#d62728")], title="rlocus(tf([1 0], [1 1 2]))")

Dcontrol1: max |y - analytic| = 7.9e-11, y(10) = 0.0001


## 6. PD 控制 `PDcontrol1.mdl`

Step（$t=0$）→ Sum（$e=r-y$）→ 兩路：Gain 100（P）與 $K_d=10$ → Derivative（D），相加後送進 $\dfrac{1}{s^2+s+2}$，模擬 10 秒。
Step time 是 0，Derivative 看不到設定值的跳變，D 項等於 $-10\dot y$：
$\ddot y+11\dot y+102y=100$，穩態值 $\dfrac{100}{102}=0.9804$、$\zeta=\dfrac{11}{2\sqrt{102}}=0.545$。
同樣 $K_p=100$ 但沒有 D 項時 $\zeta=\dfrac{1}{2\sqrt{102}}=0.050$，振盪劇烈。

In [7]:
def pd(Kp, Kd):
    def rhs(t, x):
        e = step(t, 0) - Gp.y(x)
        return Gp.dx(x, Kp * e - Kd * Gp.dy(x))                       # Sum2 "+|+"：P 與 D 兩路相加
    t, X = simulate(rhs, np.zeros(2), 10)
    return t, Gp.C @ X


panels = {"Scope": []}
for Kd in (10, 0):
    t, y = pd(100, Kd)
    zeta = (1 + Kd) / (2 * np.sqrt(102))
    Mp = 100 * np.exp(-zeta * np.pi / np.sqrt(1 - zeta**2))
    print(f"PDcontrol1 Kp = 100, Kd = {Kd:2d}: zeta = {zeta:.3f}, overshoot = {overshoot(y, 100 / 102):.1f} % (theory {Mp:.1f} %), y(10) = {y[-1]:.4f}")
    panels["Scope"].append((f"Kp = 100, Kd = {Kd}" + ("" if Kd else " (P only, for comparison)"), y))
scope(t, panels, "PDcontrol1.mdl: derivative action adds damping", h=420)
rlocus([1, 10], [1, 1, 2], np.linspace(0, 30, 601), marks=[(10, "#d62728")],
       title="PD control: rlocus of Kd (s + Kp/Kd) / (s^2+s+2), Kp/Kd = 10")

PDcontrol1 Kp = 100, Kd = 10: zeta = 0.545, overshoot = 13.0 % (theory 13.0 %), y(10) = 0.9804


PDcontrol1 Kp = 100, Kd =  0: zeta = 0.050, overshoot = 85.6 % (theory 85.6 %), y(10) = 0.9741


PD 控制器 $K_d(s+K_p/K_d)$ 在 $-10$ 加了一個零點，把根軌跡往左拉成一個圓弧，閉迴路極點遠離虛軸、阻尼變大：
和網頁「Deforming of the root locus to the left by an additional zero」的說明一致。